In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from collections import Counter

# Setting Device (Gunakan GPU jika ada, jika tidak pakai CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Perangkat yang digunakan: {device}")

# 1. Load Data Bersih
path_data = '../DATASETUAP/data_bersih.csv'
df = pd.read_csv(path_data)

# Pastikan tidak ada data kosong
df.dropna(inplace=True)

print(f"Jumlah Data: {len(df)}")
display(df.head())

Perangkat yang digunakan: cpu
Jumlah Data: 13169


,text_clean,label
0,disaat semua cowok berusaha melacak perhatian ...,1
1,rt user user siapa telat ngasih tau eluedan sa...,1
2,kadang aku berfikir aku tetap percaya tuhan pa...,0
3,user user aku akunnku tau matamu sipit diliat ...,0
4,user user kaum cebong kapir udah keliatan dong...,1


In [2]:
# ==========================================
# PERSIAPAN DATA (TOKENISASI)
# ==========================================

# 1. Gabungkan semua teks untuk membangun kamus kata
all_text = ' '.join(df['text_clean'].values)
words = all_text.split()

# 2. Hitung frekuensi kata (ambil 5000 kata terpopuler saja biar ringan)
count_words = Counter(words)
total_words = len(words)
sorted_words = count_words.most_common(total_words)

# 3. Buat Kamus (Vocab)
vocab_to_int = {w:i+1 for i, (w,c) in enumerate(sorted_words)}

# 4. Fungsi mengubah kalimat jadi angka
def encode_text(text):
    words = text.split()
    encoded = [vocab_to_int.get(w, 0) for w in words] # 0 jika kata tidak ada di kamus
    return encoded

# Terapkan ke data
df['encoded'] = df['text_clean'].apply(encode_text)

# 5. Padding (Menyamakan panjang kalimat)
# Kita set panjang kalimat fix 100 kata. Jika kurang ditambah 0, jika lebih dipotong.
seq_length = 100

def pad_features(encoded_reviews, seq_length):
    features = np.zeros((len(encoded_reviews), seq_length), dtype=int)
    for i, row in enumerate(encoded_reviews):
        features[i, -len(row):] = np.array(row)[:seq_length]
    return features

X = pad_features(df['encoded'].values, seq_length)
y = df['label'].values

print("✅ Tokenisasi & Padding Selesai!")
print(f"Contoh data pertama (Angka): {X[0][:20]} ...")

✅ Tokenisasi & Padding Selesai!
Contoh data pertama (Angka): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] ...


In [3]:
# Split Data 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ubah ke format Tensor (Format khusus PyTorch)
train_data = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_data = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

# Buat DataLoader (Untuk mengambil data per batch saat latihan)
batch_size = 50
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)

print(f"Jumlah Data Train: {len(X_train)}")
print(f"Jumlah Data Test: {len(X_test)}")

Jumlah Data Train: 10535
Jumlah Data Test: 2634


In [4]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.output_size = output_size
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        
        # Layer 1: Embedding (Mengubah angka jadi vektor)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Layer 2: LSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=drop_prob, batch_first=True)
        # Layer 3: Dropout (Mencegah overfitting)
        self.dropout = nn.Dropout(0.3)
        # Layer 4: Fully Connected
        self.fc = nn.Linear(hidden_dim, output_size)
        # Layer 5: Sigmoid (Agar outputnya 0 sampai 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        batch_size = x.size(0)
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim)
        
        out = self.dropout(lstm_out)
        out = self.fc(out)
        sig_out = self.sigmoid(out)
        
        sig_out = sig_out.view(batch_size, -1)
        sig_out = sig_out[:, -1] # Ambil output terakhir
        return sig_out, hidden

    def init_hidden(self, batch_size):
        weight = next(self.parameters()).data
        hidden = (weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device),
                  weight.new(self.n_layers, batch_size, self.hidden_dim).zero_().to(device))
        return hidden

# Inisialisasi Model
vocab_size = len(vocab_to_int) + 1 # +1 untuk padding 0
output_size = 1
embedding_dim = 200 # Dimensi vektor kata
hidden_dim = 128    # Jumlah neuron LSTM
n_layers = 2        # Tumpukan LSTM

model = SentimentLSTM(vocab_size, output_size, embedding_dim, hidden_dim, n_layers)
model.to(device)

print(model)

SentimentLSTM(
  (embedding): Embedding(31142, 200)
  (lstm): LSTM(200, 128, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [6]:
# Setting Training
lr = 0.001
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
epochs = 5 

print("🚀 Mulai Training... (Mohon bersabar)")

model.train()
for epoch in range(epochs):
    counter = 0
    
    for inputs, labels in train_loader:
        counter += 1
        
        # PERBAIKAN DISINI:
        # Kita inisialisasi hidden layer SESUAI ukuran batch saat ini (inputs.size(0))
        # Jadi kalau datanya sisa 35, dia ikut 35. Kalau 50, dia ikut 50.
        h = model.init_hidden(inputs.size(0))
        
        # Pindahkan hidden state ke device (CPU/GPU) juga
        h = tuple([each.data for each in h])
        
        inputs, labels = inputs.to(device), labels.float().to(device)
        model.zero_grad()
        
        output, h = model(inputs, h)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        if counter % 50 == 0:
            print(f"Epoch: {epoch+1}/{epochs} | Step: {counter} | Loss: {loss.item():.4f}")

print("✅ Training Selesai!")

🚀 Mulai Training... (Mohon bersabar)
Epoch: 1/5 | Step: 50 | Loss: 0.4115
Epoch: 1/5 | Step: 100 | Loss: 0.3085
Epoch: 1/5 | Step: 150 | Loss: 0.2577
Epoch: 1/5 | Step: 200 | Loss: 0.3313
Epoch: 2/5 | Step: 50 | Loss: 0.2877
Epoch: 2/5 | Step: 100 | Loss: 0.0706
Epoch: 2/5 | Step: 150 | Loss: 0.3330
Epoch: 2/5 | Step: 200 | Loss: 0.2107
Epoch: 3/5 | Step: 50 | Loss: 0.1560
Epoch: 3/5 | Step: 100 | Loss: 0.0584
Epoch: 3/5 | Step: 150 | Loss: 0.0970
Epoch: 3/5 | Step: 200 | Loss: 0.0942
Epoch: 4/5 | Step: 50 | Loss: 0.1580
Epoch: 4/5 | Step: 100 | Loss: 0.0579
Epoch: 4/5 | Step: 150 | Loss: 0.0266
Epoch: 4/5 | Step: 200 | Loss: 0.0246
Epoch: 5/5 | Step: 50 | Loss: 0.0080
Epoch: 5/5 | Step: 100 | Loss: 0.0929
Epoch: 5/5 | Step: 150 | Loss: 0.0479
Epoch: 5/5 | Step: 200 | Loss: 0.0423
✅ Training Selesai!


In [8]:
import os
import joblib

# 1. Buat folder 'models' secara otomatis jika belum ada
# Posisi folder akan sejajar dengan folder 'Code'
folder_path = '../models'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"📁 Folder '{folder_path}' berhasil dibuat!")

# 2. Simpan Model LSTM
torch.save(model.state_dict(), f'{folder_path}/model_lstm.pth')

# 3. Simpan Vocab (Kamus Kata)
joblib.dump(vocab_to_int, f'{folder_path}/vocab.pkl')

print("✅ SUKSES! Model LSTM dan Vocab tersimpan di folder 'models'.")
print("Kamu sudah menyelesaikan Model 1 (Base Model).")

📁 Folder '../models' berhasil dibuat!
✅ SUKSES! Model LSTM dan Vocab tersimpan di folder 'models'.
Kamu sudah menyelesaikan Model 1 (Base Model).
